# Bayesian Signals 

We can use bayesian updating based upon the information we have on the discards.

| Event | Signal | Update |
|---|---|---|
| Opponent passes on face-up card X | X not useful → not collecting X's melds | Downweight X's meld partners |
| Opponent discards Y | Y not useful → not collecting Y's melds | Downweight Y's meld partners |
| Opponent picks up X from discard | X IS useful → probably collecting X's melds | Upweight X's meld partners |

In [15]:
import sys, os
sys.path.insert(0, os.path.abspath('../'))
                                   
import random
import matplotlib.pyplot as plt
from agent.cards import make_deck, Card, RANK_INDEX
from agent.inference import BeliefState
from agent.eval.sim import simulate_game, Event

In [ ]:
# Deal a fixed starting hand for the demonstration cells below
import random
random.seed(42)
deck = make_deck()
random.shuffle(deck)
own_hand = deck[:10]
face_up  = deck[20]
unknown  = [c for c in make_deck() if c not in set(own_hand) and c != face_up]
print(f"Own hand: {[str(c) for c in own_hand]}")
print(f"Face up:  {face_up}")
print(f"Unknown cards: {len(unknown)}")

## 1. Meld Partners

Checks for meld partners, either cards of same number (set) or within two either side of the same suit (run)

In [16]:
bs = BeliefState(own_hand, face_up)

# Pick a card in the middle of the rank range so run partners exist on both sides
test_card = Card('7', 'H')
partners = bs._meld_partners(test_card)

set_partners = sorted([c for c in partners if c.rank == test_card.rank], key=str)
run_partners = sorted([c for c in partners if c.suit == test_card.suit], key=lambda c: RANK_INDEX[c.rank])

print(f'Card: {test_card}')
print(f'Set partners (same rank): {[str(c) for c in set_partners]}')
print(f'Run partners (same suit, within 2): {[str(c) for c in run_partners]}')
print(f'Total partners: {len(partners)}')

Card: 7H
Set partners (same rank): ['7C', '7D', '7S']
Run partners (same suit, within 2): ['5H', '6H', '8H', '9H']
Total partners: 7


## 2. Discard Signal

We can improve accuracy by using Bayesian inference to read into the opponent's strategy based on their discards. 

If an opponent discards a card, it is highly likely they are not collecting a meld (runs or sets) associated with that card. We can downweight the probabilities of any remaining unseen cards that would form a meld with it using a hyperparameter $\lambda$ (where $0 \le \lambda < 1$).

#### The Update Rule:

$$P(\text{card } i \in \text{ hand} \mid \text{discard } d) \propto P(\text{discard } d \mid \text{card } i \in \text{ hand}) \cdot P(\text{card } i \in \text{ hand})$$

We apply this sequentially as:

$$p_i \leftarrow \eta \cdot L(i \mid d) \cdot p_i$$

Where **$\eta$** is the normalization constant to keep the hand-size invariant at 10, and **$L(i \mid d)$** is the likelihood penalty:

$$L(i \mid d) = \begin{cases} \lambda & \text{if card } i \text{ forms a meld with } d \\ 1 & \text{otherwise} \end{cases}$$

In [17]:
discarded = unknown[0]   # the card the opponent discards
partners  = list(bs._meld_partners(discarded))
non_partners = [c for c in unknown[1:] if c not in set(partners)]

# Uniform baseline
bs_uni = BeliefState(own_hand, face_up)
bs_uni.observe_stock_draw_then_discard(discarded)

# Bayesian
bs_bay = BeliefState(own_hand, face_up)
bs_bay.observe_stock_draw_then_discard_bayesian(discarded, lambda_=0.4)

avg_partner_uni = sum(bs_uni.prob(c) for c in partners) / len(partners)
avg_partner_bay = sum(bs_bay.prob(c) for c in partners) / len(partners)
avg_other_uni   = sum(bs_uni.prob(c) for c in non_partners) / len(non_partners)
avg_other_bay   = sum(bs_bay.prob(c) for c in non_partners) / len(non_partners)

print(f'Discarded card: {discarded}')
print(f'Meld partners: {[str(c) for c in partners]}')
print()
print(f'                   Uniform    Bayesian')
print(f'Avg P(partner):    {avg_partner_uni:.6f}   {avg_partner_bay:.6f}')
print(f'Avg P(non-partner):{avg_other_uni:.6f}   {avg_other_bay:.6f}')
print(f'Ratio:             {avg_partner_uni/avg_other_uni:.4f}      {avg_partner_bay/avg_other_bay:.4f}')
print()
print(f'Sum (uniform):  {bs_uni.hand_size_belief:.6f}')
print(f'Sum (Bayesian): {bs_bay.hand_size_belief:.6f}')

Discarded card: AH
Meld partners: ['AD', '2H', 'AS', 'AC', '3H']

                   Uniform    Bayesian
Avg P(partner):    0.250000   0.108108
Avg P(non-partner):0.242857   0.262548
Ratio:             1.0294      0.4118

Sum (uniform):  10.000000
Sum (Bayesian): 10.000000


## 3. Pass Signal

When an opponent draws from the stock instead of taking the face-up card, they signal that the face-up card is not useful to them. We can apply the same Bayesian downweighting logic ($\mu$) to any unseen cards that would form a meld with it.

In [18]:
passed_card = face_up
pass_partners = list(bs._meld_partners(passed_card))

bs_pass = BeliefState(own_hand, face_up)
p_before = {c: bs_pass.prob(c) for c in pass_partners}
bs_pass.observe_opponent_pass_discard(passed_card, mu=0.4)
p_after = {c: bs_pass.prob(c) for c in pass_partners}

print(f'Passed card: {passed_card}')
print(f'Partners: {[str(c) for c in pass_partners]}')
print()
print(f'Partner       Before      After       Change')
for c in sorted(pass_partners, key=str):
    print(f'{str(c):12s}  {p_before[c]:.6f}  {p_after[c]:.6f}  {p_after[c]-p_before[c]:+.6f}')
print(f'\nSum: {bs_pass.hand_size_belief:.6f} (expect 10.0)')

Passed card: 9D
Partners: ['8D', 'JD', '9H', '7D', '10D', '9S', '9C']

Partner       Before      After       Change
10D           0.243902  0.105263  -0.138639
7D            0.243902  0.105263  -0.138639
8D            0.000000  0.000000  +0.000000
9C            0.243902  0.105263  -0.138639
9H            0.243902  0.105263  -0.138639
9S            0.000000  0.000000  +0.000000
JD            0.243902  0.105263  -0.138639

Sum: 10.000000 (expect 10.0)


## 4. Draw-from-Discard Signal

When the opponent picks up the face-up card $X$, it is guaranteed to be useful to them. We set $P(X) = 1.0$ and upweight its immediate meld partners with parameter $\nu$, as they are now highly likely to be residing in the opponent's hand.

In [19]:
drawn_card = face_up
draw_partners = list(bs._meld_partners(drawn_card))

bs_draw_uni = BeliefState(own_hand, face_up)
bs_draw_uni.observe_opponent_draw_discard(drawn_card)

bs_draw_bay = BeliefState(own_hand, face_up)
bs_draw_bay.observe_opponent_draw_discard_bayesian(drawn_card, nu=2.0)

print(f'Drawn card: {drawn_card}')
print(f'Partners: {[str(c) for c in sorted(draw_partners, key=str)]}')
print()
print(f'Partner       Uniform     Bayesian')
for c in sorted(draw_partners, key=str):
    print(f'{str(c):12s}  {bs_draw_uni.prob(c):.6f}  {bs_draw_bay.prob(c):.6f}')
print(f'\nSum (uniform):  {bs_draw_uni.hand_size_belief:.6f}  (expect 11.0)')
print(f'Sum (Bayesian): {bs_draw_bay.hand_size_belief:.6f}  (expect 11.0)')

Drawn card: 9D
Partners: ['10D', '7D', '8D', '9C', '9H', '9S', 'JD']

Partner       Uniform     Bayesian
10D           0.243902  0.439122
7D            0.243902  0.439122
8D            0.000000  0.000000
9C            0.243902  0.439122
9H            0.243902  0.439122
9S            0.000000  0.000000
JD            0.243902  0.439122

Sum (uniform):  11.000000  (expect 11.0)
Sum (Bayesian): 11.000000  (expect 11.0)


## 5. Combined Effect Over a Full Turn

A full turn updates our beliefs sequentially through the pass, stock-draw, and discard signals, temporarily bumping the hand-size mass to 11 before resetting to 10. Compared to a uniform model, this creates probability peaks on unrelated hidden cards and valleys on the downweighted meld partners.

In [20]:
from agent.eval.sim import simulate_game

N_TURNS = 30
SEED    = 7

result = simulate_game(n_turns=N_TURNS, seed=SEED)
face_up = result.face_up

bs_uni = BeliefState(result.starting_own_hand, face_up)
bs_bay = BeliefState(result.starting_own_hand, face_up)

snapshots_uni = []
snapshots_bay = []

opp_turn_pending  = False
opp_drew_from_stock = False  # distinguishes stock-draw from discard-draw turns

for e in result.events:
    if e.type == "opp_pass_discard":
        bs_bay.observe_opponent_pass_discard(e.card, mu=0.4)

    elif e.type == "opp_draw_discard":
        bs_uni.observe_opponent_draw_discard(e.card)
        bs_bay.observe_opponent_draw_discard_bayesian(e.card, nu=2.0)
        opp_turn_pending    = True
        opp_drew_from_stock = False

    elif e.type == "opp_draw_stock":
        opp_turn_pending    = True
        opp_drew_from_stock = True

    elif e.type == "opp_discard":
        bs_uni.observe_opponent_discard(e.card)
        if opp_drew_from_stock:
            # Full stock-draw signal: stock update + discard signal
            bs_bay.observe_stock_draw_then_discard_bayesian(e.card, lambda_=0.4)
        else:
            # Opponent drew from discard (draw already handled) — just discard signal
            bs_bay.observe_opponent_discard_bayesian(e.card, lambda_=0.4)
        opp_turn_pending    = False
        opp_drew_from_stock = False
        snapshots_uni.append(dict(bs_uni.beliefs()))
        snapshots_bay.append(dict(bs_bay.beliefs()))

    elif e.type == "own_draw_stock":
        bs_uni.observe_own_draw_stock(e.card)
        bs_bay.observe_own_draw_stock(e.card)

    elif e.type == "own_draw_discard":
        bs_uni.observe_own_draw_discard(e.card)
        bs_bay.observe_own_draw_discard(e.card)

    elif e.type == "own_discard":
        bs_uni.observe_own_discard(e.card)
        bs_bay.observe_own_discard(e.card)

# --- Plot ---
n_snaps   = len(snapshots_uni)
all_cards = make_deck()
in_opp    = [c in set(result.opp_hand) for c in all_cards]

# Alphabetical sort: suit then rank, stable across all turns
from agent.cards import SUITS, RANKS
order    = sorted(range(len(all_cards)),
                  key=lambda j: (SUITS.index(all_cards[j].suit), RANKS.index(all_cards[j].rank)))
labels_s = [str(all_cards[j]) for j in order]
in_opp_s = [in_opp[j]         for j in order]

fig, axes = plt.subplots(n_snaps, 2, figsize=(22, 3.5 * n_snaps))
if n_snaps == 1:
    axes = [axes]
fig.suptitle(
    f"Belief state after each opp turn  |  Red = actually in opp hand\n"
    f"(seed={SEED}, {result.n_turns} total turns played)",
    fontsize=12, y=1.002
)

for i, (beliefs_u, beliefs_b) in enumerate(zip(snapshots_uni, snapshots_bay)):
    probs_u = [beliefs_u[all_cards[j]] for j in order]
    probs_b = [beliefs_b[all_cards[j]] for j in order]
    x = range(len(all_cards))

    for ax, probs, base_col, subtitle in [
        (axes[i][0], probs_u, "steelblue",  f"After opp turn {i+1}  [uniform]"),
        (axes[i][1], probs_b, "darkorange",  f"After opp turn {i+1}  [Bayesian λ=0.4]"),
    ]:
        cols = ["crimson" if g else base_col for g in in_opp_s]
        ax.bar(x, probs, color=cols, width=0.8)
        ax.set_title(subtitle, fontsize=8)
        ax.set_xticks(list(x))
        ax.set_xticklabels(labels_s, rotation=90, fontsize=5)
        ax.set_ylim(0, 0.7)
        ax.set_ylabel("P", fontsize=7)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="crimson",    label="In opp hand (ground truth)"),
    Patch(facecolor="steelblue",  label="Not in opp hand (uniform)"),
    Patch(facecolor="darkorange", label="Not in opp hand (Bayesian)"),
]
axes[-1][1].legend(handles=legend_elements, loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

print(f"Opp hand (ground truth): {[str(c) for c in result.opp_hand]}")
print(f"Invariant — uniform:  {bs_uni.hand_size_belief:.6f}")
print(f"Invariant — Bayesian: {bs_bay.hand_size_belief:.6f}")
